# 00 · Audit Stage 3 data

Run this before preprocessing. It only inspects files and does not modify the dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection')
REPO = Path('/content/Blackbox-Detection')
DATA_ROOT = DRIVE_ROOT / 'DATASET'
COMMA_ROOT = DATA_ROOT / 'comma2k19'
RAW_ROOT = COMMA_ROOT / 'raw'
PROCESSED_ROOT = COMMA_ROOT / 'processed' / 'v1'
MANIFEST_ROOT = DRIVE_ROOT / 'manifests' / 'stage3' / 'v1'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs' / 'stage3'
PRETRAINED_ROOT = DRIVE_ROOT / 'pretrained'

# Clone your repository if this runtime does not have it yet.
if not REPO.exists():
    raise RuntimeError('Clone Blackbox-Detection to /content/Blackbox-Detection first, then rerun this cell.')
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('RAW_ROOT      :', RAW_ROOT)
print('PROCESSED_ROOT:', PROCESSED_ROOT)
# Install this repository through its existing pyproject.toml without replacing
# Colab's binary stack. Dependency versions in pyproject.toml are aligned to the
# DACON evaluation-server package list.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)],
    check=True,
)


In [ ]:
from blackbox_detection.stage3.comma2k19 import find_archives, discover_segments

archives = find_archives(RAW_ROOT)
print('archives:', len(archives))
for p in archives:
    print(f'{p.name:20s} {p.stat().st_size / 2**30:7.2f} GiB')
assert archives, f'No Chunk zip files found under {RAW_ROOT}'

In [ ]:
import zipfile

first = archives[0]
refs = discover_segments(first)
print('first archive:', first.name)
print('segments     :', len(refs))
print('first ref    :', refs[0])

with zipfile.ZipFile(first) as zf:
    names = zf.namelist()
    prefix = refs[0].prefix.lower() + '/'
    sample = [n for n in names if n.lower().startswith(prefix)]
    for n in sample[:80]:
        print(n)

Expected: each segment should contain `video.hevc`, `processed_log/...car_speed...`, `...steering_angle...`, IMU files, and a global pose/frame-times array. If those paths look substantially different, stop before running notebook 01.